# Titanic Survival Prediction Model

End-to-end workflow to explore the Titanic dataset, train a scikit-learn `RandomForestClassifier` pipeline with preprocessing, evaluate validation metrics, and export the trained model artifact.

In [ ]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("../data/train.csv")
print(df.shape)
df.head()

In [ ]:
print("Missing values:\n", df.isna().sum())
print("\nOverall survival rate:", round(df["Survived"].mean(), 4))
print("\nSurvival rate by Sex and Pclass:\n", df.groupby(["Sex", "Pclass"])["Survived"].mean().unstack())

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features]
y = df["Survived"]

numeric_cols = ["Age", "SibSp", "Parch", "Fare"]
categorical_cols = ["Pclass", "Sex", "Embarked"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_cols,
        ),
    ]
)

clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=7,
                random_state=42,
            ),
        ),
    ]
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
preds = clf.predict(X_val)
probas = clf.predict_proba(X_val)[:, 1]

print(f"Validation Accuracy: {accuracy_score(y_val, preds):.4f}")
print(f"Validation ROC AUC:  {roc_auc_score(y_val, probas):.4f}")
print(classification_report(y_val, preds))

In [ ]:
import os
os.makedirs("../model", exist_ok=True)
joblib.dump(clf, "../model/titanic_model.joblib")
print("Saved trained pipeline to ../model/titanic_model.joblib")